
# Q3(b) — Rule-Based Baseline for ESG Message Triage

**Unit:** BUS5001 — Cloud Platforms and Analytics  
**Assessment:** Assessment 3 — Robotic Process Automation, AI and Cloud-Enabled ESG Analytics  
**Experiment:** Rule-Based Classification and LLM Comparison  
**Platform:** Google Colab (Python)  
**LLM used for comparison:** GPT-4.1 mini (Microsoft Foundry)

## 1. Experiment Objective

The purpose of this experiment is to compare the ESG message classifications generated by GPT-4.1 mini with a simple rule-based baseline.

In the previous Microsoft Foundry experiment, three fictional employee messages were classified using an improved prompt. The messages covered energy efficiency, accessibility and supplier governance.

For this comparison, I developed a Python classifier using predefined keywords and urgency conditions. I then compared its outputs with the actual classifications recorded from GPT-4.1 mini.

The comparison focuses on four fields: issue category, urgency, recommended team and data sensitivity. The aim is to examine consistency between the two approaches and identify their limitations, rather than claim that either method is fully accurate.



## 2. Test Messages and Baseline Rules

Three fictional ESG-related messages were used in this experiment:

| Test ID | ESG issue | Message summary |
|---|---|---|
| T01 | Energy efficiency | Air conditioning running overnight in an empty office |
| T02 | Accessibility | Blocked accessible entrance preventing wheelchair access |
| T03 | Procurement governance | Suspected supplier certificate falsification and confidentiality concerns |

### Rule-Based Classification Logic

The baseline applies the following rules:

- **R1 — Energy:** Messages containing "air conditioning", "electricity" or "energy" are classified as ENERGY and assigned to Facilities/Energy. Ongoing issues are assigned MEDIUM urgency.
- **R2 — Accessibility:** Messages containing "accessible entrance", "wheelchair" or "accessibility" are classified as ACCESSIBILITY and assigned to Facilities/Accessibility Officer. Blocked access is assigned HIGH urgency.
- **R3 — Procurement governance:** Messages containing "supplier", "sustainability certificate" or "procurement" are classified as PROCUREMENT_GOVERNANCE and assigned to Procurement/Compliance. Suspected misconduct is assigned HIGH urgency.
- **R4 — Confidentiality:** Messages containing "confidential", "retaliation" or "whistleblower" are assigned HIGH data sensitivity without changing the existing category.
- **R5 — Other:** Messages that do not match the category rules are classified as OTHER and assigned to ESG Operations, with provisional LOW urgency pending human review.

The rules are deliberately simple so that the classification decisions can be explained and reproduced. They were developed around the selected examples, so their performance on unfamiliar messages remains uncertain.


In [ ]:

import pandas as pd

# Q3(b): Rule-based ESG message classification
# BUS5001 Assessment 3
# Three fictional messages used in the Microsoft Foundry experiment

messages = {
    "T01": "The air conditioning has been running overnight in an empty office for the past week.",
    "T02": "The accessible entrance to Building A is blocked, and a wheelchair user cannot enter the building.",
    "T03": "I think one of our suppliers may be falsifying its sustainability certificates. Please keep my name confidential because I am worried about retaliation."
}

# Actual GPT-4.1 mini classifications recorded in Microsoft Foundry
llm_results = {
    "T01": {
        "category": "ENERGY",
        "urgency": "MEDIUM",
        "team": "Facilities/Energy",
        "sensitivity": "LOW"
    },
    "T02": {
        "category": "ACCESSIBILITY",
        "urgency": "HIGH",
        "team": "Facilities/Accessibility Officer",
        "sensitivity": "LOW"
    },
    "T03": {
        "category": "PROCUREMENT_GOVERNANCE",
        "urgency": "HIGH",
        "team": "Procurement/Compliance",
        "sensitivity": "HIGH"
    }
}

def classify_esg(message):
    text = message.lower()

    # R1-R3: Category and team classification
    if any(word in text for word in
           ["air conditioning", "electricity", "energy"]):
        category = "ENERGY"
        team = "Facilities/Energy"
        urgency = "MEDIUM" if any(
            word in text for word in
            ["overnight", "past week", "ongoing"]
        ) else "LOW"

    elif any(word in text for word in
             ["accessible entrance", "wheelchair", "accessibility"]):
        category = "ACCESSIBILITY"
        team = "Facilities/Accessibility Officer"
        urgency = "HIGH" if any(
            word in text for word in
            ["blocked", "cannot enter", "unable to access"]
        ) else "LOW"

    elif any(word in text for word in
             ["supplier", "sustainability certificate", "procurement"]):
        category = "PROCUREMENT_GOVERNANCE"
        team = "Procurement/Compliance"
        urgency = "HIGH" if any(
            word in text for word in
            ["falsifying", "misconduct", "fraud"]
        ) else "LOW"

    else:
        # R5: No category keywords found
        category = "OTHER"
        team = "ESG Operations"
        urgency = "LOW"

    # R4: Confidentiality override
    sensitivity = "HIGH" if any(
        word in text for word in
        ["confidential", "retaliation", "whistleblower"]
    ) else "LOW"

    return {
        "category": category,
        "urgency": urgency,
        "team": team,
        "sensitivity": sensitivity
    }


# Run baseline and compare with recorded LLM classifications
rows = []

for test_id, message in messages.items():
    baseline = classify_esg(message)
    llm = llm_results[test_id]

    for field in ["category", "urgency", "team", "sensitivity"]:
        rows.append({
            "Test": test_id,
            "Field": field,
            "GPT-4.1 mini": llm[field],
            "Rule-based baseline": baseline[field],
            "Match": "Yes" if llm[field] == baseline[field] else "No"
        })

comparison_df = pd.DataFrame(rows)

print("Q3(b) — GPT-4.1 mini vs Rule-Based Baseline")
print("=" * 55)

display(comparison_df)

matches = (comparison_df["Match"] == "Yes").sum()
total = len(comparison_df)

print(f"\nAgreement: {matches}/{total} fields")
print(f"Agreement percentage: {matches / total * 100:.1f}%")
print("\nNote: Agreement is not the same as classification accuracy.")

# Save comparison evidence for GitHub
comparison_df.to_csv("Q3_baseline_comparison.csv", index=False)

print("\nCSV saved: Q3_baseline_comparison.csv")


Q3(b) — GPT-4.1 mini vs Rule-Based Baseline


,Test,Field,GPT-4.1 mini,Rule-based baseline,Match
0,T01,category,ENERGY,ENERGY,Yes
1,T01,urgency,MEDIUM,MEDIUM,Yes
2,T01,team,Facilities/Energy,Facilities/Energy,Yes
3,T01,sensitivity,LOW,LOW,Yes
4,T02,category,ACCESSIBILITY,ACCESSIBILITY,Yes
5,T02,urgency,HIGH,HIGH,Yes
6,T02,team,Facilities/Accessibility Officer,Facilities/Accessibility Officer,Yes
7,T02,sensitivity,LOW,LOW,Yes
8,T03,category,PROCUREMENT_GOVERNANCE,PROCUREMENT_GOVERNANCE,Yes
9,T03,urgency,HIGH,HIGH,Yes



Agreement: 12/12 fields
Agreement percentage: 100.0%

Note: Agreement is not the same as classification accuracy.

CSV saved: Q3_baseline_comparison.csv



## 3. Results and Interpretation

The Python baseline produced classifications for all three ESG messages. The results were compared with the actual GPT-4.1 mini outputs recorded in Microsoft Foundry.

### Comparison Results

| Test | Category | Urgency | Recommended team | Sensitivity | Agreement |
|---|---|---|---|---|---|
| T01 | ENERGY | MEDIUM | Facilities/Energy | LOW | All four fields matched |
| T02 | ACCESSIBILITY | HIGH | Facilities/Accessibility Officer | LOW | All four fields matched |
| T03 | PROCUREMENT_GOVERNANCE | HIGH | Procurement/Compliance | HIGH | All four fields matched |

**Overall agreement: 12/12 fields (100%).**

The two approaches produced matching classifications for the selected messages. This suggests that the revised LLM prompt generated results consistent with the predefined baseline rules for these examples.

However, this result should not be interpreted as 100% classification accuracy. The baseline was designed using the same three scenarios, and the experiment did not include an independent human-labelled test dataset.

The rule-based approach is predictable because it uses explicit keyword conditions. In contrast, an LLM can potentially interpret messages that use different wording, although this advantage was not independently tested here.



## 4. Limitations and Conclusion

### Limitations

This experiment has several limitations:

1. **Small sample:** Only three fictional messages were evaluated.
2. **Keyword dependency:** The baseline may miss issues expressed using unfamiliar words or indirect language.
3. **Limited validation:** Agreement with the baseline does not establish accuracy against independent ground-truth labels.
4. **Potential bias:** The tests do not establish whether either method performs consistently across different writing styles, languages or sensitive contexts.
5. **Simplified rules:** The baseline does not generate sentiment labels or detailed escalation explanations, so these fields were not included in the comparison.

### Conclusion

The Python baseline and GPT-4.1 mini produced matching classifications for the three selected ESG messages. The comparison provides a reproducible check of the revised prompt's outputs, but it is not sufficient evidence for production readiness.

Before operational deployment, I would recommend testing a larger and more varied dataset, validating classifications against human-reviewed labels, introducing automated JSON validation and retaining human oversight for urgent or confidential issues.

This notebook provides supporting evidence for the Q3(b) evaluation documented in the assessment report and GitHub experiment log.
